In [1]:
import os
import re
import random
import numpy as np
import pandas as pd
import mne
import itertools
from scipy.stats import kurtosis
from scipy.signal import welch
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score
import tensorflow as tf
import xgboost as xgb
from transformers import Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset

In [2]:
# Constants
FILENAME_PATTERN = re.compile(r'^(?P<subject>s\d{2})_(?P<experiment>ex\d{2})(?P<session>_s\d{2})?.csv$')
DATA_PATH = "/kaggle/input/data-eeg/auditory-evoked-potential-eeg-biometric-dataset-1.0.0"
ELECTRODES = ["T7", "F8", "Cz", "P4"]
SFREQ = 200
BANDS = {"delta": (0.5, 4), "theta": (4, 8), "alpha": (8, 13), "beta": (13, 30), "gamma": (30, 40)}

In [3]:
# Parse filename
def parse_filename(filename):
    match = FILENAME_PATTERN.match(filename)
    if not match:
        return None
    info = match.groupdict()
    info['subject'] = int(info['subject'][1:])
    info['experiment'] = int(info['experiment'][2:])
    info['session'] = int(info['session'][2:]) if info['session'] else None
    return info

In [4]:
def load_eeg_data(data_path=DATA_PATH, subfolder='Segmented_Data'):
    folder_path = os.path.join(data_path, subfolder)
    eeg_data, eeg_df = {}, {}
    
    for file in os.listdir(folder_path):
        if file.endswith('.csv'):
            file_info = parse_filename(file)
            if not file_info:
                continue
            subject, experiment = file_info['subject'], file_info['experiment']
            file_path = os.path.join(folder_path, file)
            df = pd.read_csv(file_path)[ELECTRODES].T.values  
            
            info = mne.create_info(ch_names=ELECTRODES, sfreq=SFREQ, ch_types='eeg')
            raw = mne.io.RawArray(df, info, verbose=False)
            
            if subject not in eeg_data:
                eeg_data[subject] = {}
                eeg_df[subject] = {}
            
            if experiment not in eeg_data[subject]:
                eeg_data[subject][experiment] = []
                eeg_df[subject][experiment] = []
            
            eeg_data[subject][experiment].append(raw)
            eeg_df[subject][experiment].append(df)
    
    return eeg_data, eeg_df

In [5]:
data, df_data = load_eeg_data()
print(f"Number of subjects: {len(df_data)}")  
for subject, experiments in df_data.items():
    print(f"Subjects {subject}: {len(experiments)}")

Number of subjects: 20
Subjects 19: 8
Subjects 5: 8
Subjects 12: 8
Subjects 6: 8
Subjects 1: 8
Subjects 13: 8
Subjects 17: 8
Subjects 16: 8
Subjects 10: 8
Subjects 2: 8
Subjects 15: 8
Subjects 7: 8
Subjects 20: 8
Subjects 18: 8
Subjects 11: 8
Subjects 4: 8
Subjects 3: 8
Subjects 14: 8
Subjects 8: 8
Subjects 9: 8


In [6]:
# Preprocess EEG data
def preprocess_eeg_data(eeg_data):
    preprocessed_data = {}
    for subject in eeg_data:
        preprocessed_data[subject] = {}
        for experiment in eeg_data[subject]:
            preprocessed_experiments = []
            for raw in eeg_data[subject][experiment]:
                raw.filter(1., 40., fir_design='firwin', verbose=False)
                raw.resample(SFREQ, verbose=False)
                preprocessed_experiments.append(raw)
            preprocessed_data[subject][experiment] = preprocessed_experiments
    return preprocessed_data

In [7]:
processed_data = preprocess_eeg_data(data)

In [8]:
# Convert preprocessed data to numpy format
def convert_processed_to_df(processed_data):
    processed_df = {}
    for subject, experiments in processed_data.items():
        processed_df[subject] = {}
        for experiment, raws in experiments.items():
            processed_df[subject][experiment] = [raw.get_data() for raw in raws]
    return processed_df

In [9]:
processed_df = convert_processed_to_df(processed_data)

In [10]:
# Extract features for ML
def extract_features(epoch):
    features = []
    for channel in epoch:
        features.append(np.std(channel))
        features.append(np.ptp(channel))
        features.append(kurtosis(channel))
        f, Pxx = welch(channel, fs=SFREQ, nperseg=min(SFREQ, len(channel)))
        for band, (low, high) in BANDS.items():
            idx = np.logical_and(f >= low, f <= high)
            features.append(np.mean(Pxx[idx]) if np.any(idx) else 0)
            features.append(np.std(Pxx[idx]) if np.any(idx) else 0)
    return np.array(features)

In [11]:
def prepare_dataset(eeg_df, window_size=SFREQ*4, step_size=SFREQ*2):
    X_ml, X_dl, y_ml, y_dl = [], [], [], []
    print(f"Total subjects: {len(processed_df)}")
    
    for subject, sessions in eeg_df.items():
        all_sessions = list(itertools.chain.from_iterable(sessions.values()))
        print(f"Subject {subject}: {len(all_sessions)} sessions")
        
        for i, session in enumerate(all_sessions):
            # print(f"  Session {i}: shape={session.shape}")
            n_samples = session.shape[1]
            for start in range(0, n_samples - window_size + 1, step_size):
                epoch = session[:, start:start + window_size]
                # ML features
                features = extract_features(epoch)
                if not np.isnan(features).any():
                    X_ml.append(features)
                    y_ml.append(subject)
                # DL raw data
                X_dl.append(epoch.T)  # (timesteps, channels)
                y_dl.append(subject)
    
    X_ml = np.array(X_ml)  # (n_epochs, n_features)
    X_dl = np.array(X_dl)  # (n_epochs, window_size, channels)
    y_ml = np.array(y_ml)
    y_dl = np.array(y_dl)
    print(f"ML data shape: {X_ml.shape}, DL data shape: {X_dl.shape}")
    return X_ml, y_ml, X_dl, y_dl

In [12]:
X_ml, y_ml, X_dl, y_dl = prepare_dataset(processed_df)
print(X_ml.shape, X_dl.shape)

Total subjects: 20
Subject 19: 12 sessions
Subject 5: 12 sessions
Subject 12: 12 sessions
Subject 6: 12 sessions
Subject 1: 12 sessions
Subject 13: 12 sessions
Subject 17: 12 sessions
Subject 16: 12 sessions
Subject 10: 12 sessions
Subject 2: 12 sessions
Subject 15: 12 sessions
Subject 7: 12 sessions
Subject 20: 12 sessions
Subject 18: 12 sessions
Subject 11: 12 sessions
Subject 4: 12 sessions
Subject 3: 12 sessions
Subject 14: 12 sessions
Subject 8: 12 sessions
Subject 9: 12 sessions
ML data shape: (14234, 52), DL data shape: (14234, 800, 4)
(14234, 52) (14234, 800, 4)


In [13]:
# Define ResNet block
def resnet_block(inputs, filters, kernel_size, strides=1):
    x = tf.keras.layers.Conv1D(filters, kernel_size, strides=strides, padding='same', activation='relu')(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Conv1D(filters, kernel_size, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    
    if strides > 1 or inputs.shape[-1] != filters:
        shortcut = tf.keras.layers.Conv1D(filters, 1, strides=strides, padding='same')(inputs)
        shortcut = tf.keras.layers.BatchNormalization()(shortcut)
    else:
        shortcut = inputs
    
    x = tf.keras.layers.Add()([x, shortcut])
    x = tf.keras.layers.Activation('relu')(x)
    return x

In [14]:
# Train and evaluate models
def train_and_evaluate_models(X_ml, y_ml, X_dl, y_dl):
    X_ml_train, X_ml_test, y_ml_train, y_ml_test = train_test_split(X_ml, y_ml, test_size=0.2, random_state=42)
    X_dl_train, X_dl_test, y_dl_train, y_dl_test = train_test_split(X_dl, y_dl, test_size=0.2, random_state=42)
    
    print(f"X_ml_train shape: {X_ml_train.shape}, X_dl_train shape: {X_dl_train.shape}")
    
    unique_labels = np.unique(y_ml)
    label_map = {label: i for i, label in enumerate(unique_labels)}
    y_ml_train_mapped = np.array([label_map[label] for label in y_ml_train])
    y_ml_test_mapped = np.array([label_map[label] for label in y_ml_test])
    y_dl_train_mapped = np.array([label_map[label] for label in y_dl_train])
    y_dl_test_mapped = np.array([label_map[label] for label in y_dl_test])
    
    # ML Models
    ml_models = {
        "SVM": make_pipeline(StandardScaler(), SVC(kernel='rbf', C=1.0)),
        "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42),
        "XGBoost": xgb.XGBClassifier(max_depth=5, learning_rate=0.1, n_estimators=200, eval_metric='mlogloss', random_state=42),
        "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3))
    }
    
    results = {}
    for name, model in ml_models.items():
        print(f"\nTraining {name}...")
        model.fit(X_ml_train, y_ml_train_mapped)
        y_pred = model.predict(X_ml_test)
        accuracy = accuracy_score(y_ml_test_mapped, y_pred)
        results[name] = accuracy
        print(f"{name} Accuracy: {accuracy * 100:.2f}%")
    
    # DL Models
    n_classes = len(unique_labels)
    
    # MLP
    print("\nTraining MLP...")
    mlp_model = tf.keras.Sequential([
        tf.keras.layers.Flatten(input_shape=(X_dl_train.shape[1], X_dl_train.shape[2])),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(n_classes, activation='softmax')
    ])
    mlp_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    mlp_model.fit(X_dl_train, y_dl_train_mapped, epochs=20, batch_size=64, validation_split=0.2, verbose=1)
    _, mlp_accuracy = mlp_model.evaluate(X_dl_test, y_dl_test_mapped, verbose=0)
    results["MLP"] = mlp_accuracy
    print(f"MLP Accuracy: {mlp_accuracy * 100:.2f}%")
    
    # CNN
    print("\nTraining CNN...")
    cnn_model = tf.keras.Sequential([
        tf.keras.layers.Conv1D(32, kernel_size=3, activation='relu', input_shape=(X_dl_train.shape[1], X_dl_train.shape[2])),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Conv1D(64, kernel_size=3, activation='relu'),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(n_classes, activation='softmax')
    ])
    cnn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    cnn_model.fit(X_dl_train, y_dl_train_mapped, epochs=20, batch_size=64, validation_split=0.2, verbose=1)
    _, cnn_accuracy = cnn_model.evaluate(X_dl_test, y_dl_test_mapped, verbose=0)
    results["CNN"] = cnn_accuracy
    print(f"CNN Accuracy: {cnn_accuracy * 100:.2f}%")
    
    # LSTM
    print("\nTraining LSTM...")
    lstm_model = tf.keras.Sequential([
        tf.keras.layers.LSTM(64, input_shape=(X_dl_train.shape[1], X_dl_train.shape[2]), return_sequences=True),
        tf.keras.layers.LSTM(32),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(n_classes, activation='softmax')
    ])
    lstm_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    lstm_model.fit(X_dl_train, y_dl_train_mapped, epochs=30, batch_size=64, validation_split=0.2, verbose=1)
    _, lstm_accuracy = lstm_model.evaluate(X_dl_test, y_dl_test_mapped, verbose=0)
    results["LSTM"] = lstm_accuracy
    print(f"LSTM Accuracy: {lstm_accuracy * 100:.2f}%")
    
    # ResNet
    print("\nTraining ResNet...")
    inputs = tf.keras.Input(shape=(X_dl_train.shape[1], X_dl_train.shape[2]))  # (800, 4)
    x = tf.keras.layers.Conv1D(64, 7, strides=2, padding='same', activation='relu')(inputs)  # (400, 64)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=3, strides=2, padding='same')(x)  # (200, 64)
    
    # Stage 1: 2 blocks, 64 filters
    x = resnet_block(x, 64, 3)  # (200, 64)
    x = resnet_block(x, 64, 3)  # (200, 64)
    
    # Stage 2: 2 blocks, 128 filters
    x = resnet_block(x, 128, 3, strides=2)  # (100, 128)
    x = resnet_block(x, 128, 3)  # (100, 128)
    
    # Stage 3: 2 blocks, 256 filters
    x = resnet_block(x, 256, 3, strides=2)  # (50, 256)
    x = resnet_block(x, 256, 3)  # (50, 256)
    
    # Stage 4: 2 blocks, 512 filters
    x = resnet_block(x, 512, 3, strides=2)  # (25, 512)
    x = resnet_block(x, 512, 3)  # (25, 512)
    
    x = tf.keras.layers.GlobalAveragePooling1D()(x)  # (512,)
    outputs = tf.keras.layers.Dense(n_classes, activation='softmax')(x)  # (20,)
    
    resnet_model = tf.keras.Model(inputs, outputs)
    resnet_model.compile(optimizer='adam', 
                        loss='sparse_categorical_crossentropy', 
                        metrics=['accuracy'])
    resnet_model.fit(X_dl_train, y_dl_train_mapped, 
                     epochs=20, 
                     batch_size=64, 
                     validation_split=0.2, 
                     verbose=1,
                     callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)])
    
    _, resnet_accuracy = resnet_model.evaluate(X_dl_test, y_dl_test_mapped, verbose=0)
    results["ResNet-18"] = resnet_accuracy
    print(f"ResNet Accuracy: {resnet_accuracy * 100:.2f}%")
    
    return results

In [15]:
results = train_and_evaluate_models(X_ml, y_ml, X_dl, y_dl)

X_ml_train shape: (11387, 52), X_dl_train shape: (11387, 800, 4)

Training SVM...
SVM Accuracy: 82.82%

Training Random Forest...
Random Forest Accuracy: 87.60%

Training XGBoost...
XGBoost Accuracy: 92.66%

Training KNN...
KNN Accuracy: 81.38%

Training MLP...
Epoch 1/20
143/143 [==============================] - 5s 6ms/step - loss: 5.3384 - accuracy: 0.0571 - val_loss: 3.0018 - val_accuracy: 0.0540
Epoch 2/20
143/143 [==============================] - 1s 4ms/step - loss: 3.0342 - accuracy: 0.0743 - val_loss: 2.9957 - val_accuracy: 0.0615
Epoch 3/20
143/143 [==============================] - 1s 4ms/step - loss: 3.0356 - accuracy: 0.0765 - val_loss: 2.9955 - val_accuracy: 0.0628
Epoch 4/20
143/143 [==============================] - 1s 4ms/step - loss: 2.9802 - accuracy: 0.0858 - val_loss: 2.9944 - val_accuracy: 0.0676
Epoch 5/20
143/143 [==============================] - 1s 4ms/step - loss: 2.9367 - accuracy: 0.0909 - val_loss: 2.9855 - val_accuracy: 0.0667
Epoch 6/20
143/143 [========

In [16]:
for name, accuracy in results.items():
    print(f"{name}: {accuracy * 100:.2f}%")

SVM: 82.82%
Random Forest: 87.60%
XGBoost: 92.66%
KNN: 81.38%
MLP: 10.33%
CNN: 5.06%
LSTM: 51.95%
ResNet-18: 96.38%
